# LANGFENS — RQ1: P-SUP T-Op Official Training on Kaggle Tesla T4x2

Private execution notebook under CC BY-NC 4.0.
Executes strictly on authorized remote GPU hardware (Tesla T4) under CUDA FP16 AMP.
Strict fallback prohibition: Never fall back to local CPU/GPU training.
Capped at exactly 3,600 successful optimizer updates with per-epoch atomic remote mirroring.

## Experimental Arm: Subtype Treatment T-OP
- Objective: `operation_ce` (3-way Cross-Entropy: correct=0, substitution=1, deletion=2) + realized CTC loss
- Capacity: Exactly 266,667 parameters in `CompactOperationHead` matching control
- Partitions:
  - FIT (1,799 utterances across 12 speakers): Training gradient updates
  - SELECT (300 utterances across 2 speakers: MBMPS, TLV): Checkpoint selection & validation AP
  - CAL (300 utterances across 2 speakers: NCC, THV): Calibration
  - VAL (300 utterances across 2 speakers: NJS, RRBI): Preserved strictly unseen for evaluation
- Remote Target: `minkhoaa/langfens-rq1-cross-fold-checkpoints` under prefix `psup/t_op/`


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Setup PYTHONPATH, and Execute Training
import hashlib, os, subprocess, sys
from pathlib import Path

# Execution Gate: Set to True for immediate training execution upon Save & Run All
EXECUTE_TRAINING = True

if not EXECUTE_TRAINING:
    print("=" * 80)
    print("[HOLD] Execution gate EXECUTE_TRAINING is False.")
    print("[HOLD] Training execution is held pending explicit operator authorization.")
    print("=" * 80)
    sys.exit(0)

# 1. Clone/stage repository and checkout training branch
repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

PINNED_COMMIT_SHA = "dd2dd8953fc5d946c0b88d72e915be53e7cd0d9f"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA, f"FAIL-CLOSED: Checked out commit {actual_head} != expected {PINNED_COMMIT_SHA}"

# 2. Setup PYTHONPATH
sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"

# 3. Robust fail-closed dataset discovery
EXPECTED_ARCHIVE_SIZE = 761922918
EXPECTED_ARCHIVE_SHA256 = "b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9"

candidate_mount_locations = [
    Path("/kaggle/input/datasets/minkhoaa/langfens-rq1-fold1-staging"),
    Path("/kaggle/input/langfens-rq1-fold1-staging"),
]
if Path("/kaggle/input").exists():
    candidate_mount_locations.extend(Path("/kaggle/input").glob("**/langfens-rq1-fold1-staging"))
    candidate_mount_locations.extend(Path("/kaggle/input").glob("*fold1-staging*"))
    candidate_mount_locations.extend(Path("/kaggle/input").glob("*langfens-rq1-fold1*"))

# Deduplicate candidate directories while preserving order
seen_candidates = set()
unique_candidate_dirs = []
for p in candidate_mount_locations:
    try:
        resolved = p.resolve()
    except Exception:
        resolved = p
    if resolved not in seen_candidates:
        seen_candidates.add(resolved)
        unique_candidate_dirs.append(p)

print(f"[DATASET] Checking candidate mount locations ({len(unique_candidate_dirs)} candidates)...")
staging_dir = None
for cand in unique_candidate_dirs:
    if cand.is_dir():
        staging_dir = cand
        print(f"[DATASET] Found mounted staging directory: {staging_dir}")
        break

# Locate authorized archive
verified_archive = None
search_dirs = [staging_dir] if staging_dir is not None else unique_candidate_dirs

for sdir in search_dirs:
    if not sdir or not sdir.is_dir():
        continue
    # Priority 1: Exact filenames in staging directory
    for name in ("authorized_RQ1-TRN-02A_fold1_train_val_only.bin", "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"):
        f = sdir / name
        if f.is_file():
            verified_archive = f
            staging_dir = sdir
            break
    if verified_archive:
        break

    # Priority 2: Recursive search by exact name
    for name in ("authorized_RQ1-TRN-02A_fold1_train_val_only.bin", "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"):
        matches = list(sdir.rglob(name))
        if matches:
            verified_archive = matches[0]
            staging_dir = sdir
            break
    if verified_archive:
        break

    # Priority 3: Any *.bin / *.zip matching exact size 761,922,918 bytes
    for ext in ("*.bin", "*.zip"):
        for p in sdir.rglob(ext):
            try:
                if p.is_file() and p.stat().st_size == EXPECTED_ARCHIVE_SIZE:
                    verified_archive = p
                    staging_dir = sdir
                    break
            except OSError:
                pass
        if verified_archive:
            break
    if verified_archive:
        break

# Broad fallback across /kaggle/input if not yet found
if verified_archive is None and Path("/kaggle/input").exists():
    for ext in ("*authorized_RQ1-TRN-02A_fold1_train_val_only*", "*.bin", "*.zip"):
        for p in Path("/kaggle/input").rglob(ext):
            try:
                if p.is_file() and (
                    p.name in ("authorized_RQ1-TRN-02A_fold1_train_val_only.bin", "authorized_RQ1-TRN-02A_fold1_train_val_only.zip")
                    or p.stat().st_size == EXPECTED_ARCHIVE_SIZE
                ):
                    verified_archive = p
                    staging_dir = p.parent
                    break
            except OSError:
                pass
        if verified_archive:
            break

if verified_archive is None or not verified_archive.is_file():
    print("[DATASET] FAIL-CLOSED: Dataset discovery failed. Available /kaggle/input tree:")
    if Path("/kaggle/input").exists():
        for root, dirs, files in os.walk("/kaggle/input"):
            print(f"  [WALK] {root}: dirs={dirs}, files={files}")
    else:
        print("  [WALK] /kaggle/input does not exist!")
    raise AssertionError(
        "FAIL-CLOSED: Dataset input minkhoaa/langfens-rq1-fold1-staging not mounted or authorized archive missing!"
    )

archive_size = verified_archive.stat().st_size
print(f"[DATASET] Found authorized archive candidate: {verified_archive} ({archive_size:,} bytes)")
assert archive_size == EXPECTED_ARCHIVE_SIZE, (
    f"FAIL-CLOSED: Archive size mismatch! Expected {EXPECTED_ARCHIVE_SIZE:,} bytes, got {archive_size:,} bytes"
)

# Cryptographic SHA-256 verification (strictly fail-closed)
print(f"[DATASET] Verifying SHA-256 checksum for {verified_archive}...")
hasher = hashlib.sha256()
with open(verified_archive, "rb") as f:
    while chunk := f.read(1024 * 1024 * 8):
        hasher.update(chunk)
archive_sha = hasher.hexdigest()
print(f"[DATASET] Computed SHA-256: {archive_sha}")
assert archive_sha == EXPECTED_ARCHIVE_SHA256, (
    f"FAIL-CLOSED: Archive SHA-256 mismatch!\n"
    f"  Expected: {EXPECTED_ARCHIVE_SHA256}\n"
    f"  Actual:   {archive_sha}\n"
    f"Never fall back to another dataset!"
)
print(f"[DATASET] Cryptographic verification PASSED: {verified_archive} is authentic and authorized.")

# 4. Prepare local output directory
output_dir = Path("/kaggle/working/checkpoints/RQ1-P-SUP/t_op")
output_dir.mkdir(parents=True, exist_ok=True)

# 5. Execute Training via audit_scripts/train_p_sup.py
train_cmd = [
    sys.executable,
    "audit_scripts/train_p_sup.py",
    "--arm", "treatment",
    "--objective", "operation_ce",
    "--split-manifest", "manifests/RQ1-SPLIT-01/l2arctic_4fold_splits.json",
    "--zip-path", str(verified_archive),
    "--output-dir", str(output_dir),
    "--seed", "1337",
    "--batch-size", "2",
    "--gradient-accumulation-steps", "2",
    "--warmup-updates", "100",
    "--max-updates", "3600",
    "--lr", "0.001",
    "--weight-decay", "0.01",
    "--gradient-clip", "1.0",
    "--max-epochs", "25",
    "--train-partition", "fit",
    "--select-partition", "selection",
    "--cal-partition", "calibration",
    "--val-partition", "val",
    "--remote-execution",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
    "--remote-mirror-backend", "huggingface",
]

print("=" * 80)
print("[LAUNCH] Executing P-SUP Treatment T-OP Training:")
print("  " + " ".join(train_cmd))
print("=" * 80)
subprocess.run(train_cmd, check=True)
print("[LAUNCH] Treatment T-OP Training completed successfully.")


In [ ]:
# Phase 4: Post-Training Durability Verification & Remote Mirror Readback
import hashlib, os
from pathlib import Path
from huggingface_hub import HfApi

output_dir = Path("/kaggle/working/checkpoints/RQ1-P-SUP/t_op")
remote_repo = "minkhoaa/langfens-rq1-cross-fold-checkpoints"
remote_prefix = "psup/t_op"

api = HfApi(token=os.environ["HF_TOKEN"])

print("=" * 80)
print(f"[DURABILITY] Verifying local artifacts in {output_dir}:")
for ckpt_name in ("fixed_budget_3600.pt", "fixed_budget_checkpoint.pt", "best.pt", "last.pt", "training_summary.json", "history.json"):
    p = output_dir / ckpt_name
    if not p.is_file():
        for sub in output_dir.rglob(ckpt_name):
            p = sub
            break
    if p.is_file():
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print(f"  [LOCAL] {ckpt_name}: {p.stat().st_size:,} bytes | SHA-256: {sha}")
        rel_path = f"{remote_prefix}/{ckpt_name}"
        print(f"  [REMOTE] Ensuring remote mirror at {remote_repo}:{rel_path}...")
        api.upload_file(
            path_or_fileobj=str(p),
            path_in_repo=rel_path,
            repo_id=remote_repo,
            repo_type="model"
        )
        print(f"  -> Confirmed mirrored: {rel_path}")
    else:
        print(f"  [NOTE] Optional/alias artifact {ckpt_name} not present.")

print("[DURABILITY] Durability readback and remote mirroring completed successfully.")
print("=" * 80)


In [ ]:
# Phase 5: Final Execution Summary & Receipt
import json
from pathlib import Path

summary_files = list(Path("/kaggle/working/checkpoints/RQ1-P-SUP/t_op").rglob("training_summary.json"))
if summary_files:
    summary_data = json.loads(summary_files[0].read_text())
    print("=" * 80)
    print("LANGFENS P-SUP T-OP TRAINING SUMMARY RECEIPT:")
    print(json.dumps(summary_data, indent=2))
    print("=" * 80)
else:
    print("[RECEIPT] Execution completed.")
